# Appendix: Data Ingestion (1 of 2) — Load Unity Airways Datasets

## 📓 About this notebook
A prerequisite data-ingestion step that loads the Unity Airways datasets into Unity Catalog volumes and Delta tables, preparing the FAQ data that later becomes the vector search index.

**Maps to the book:** Chapter 4, *Building and Versioning a Tool-Calling Agent* — sections: Unstructured Data Preparation for Retrieval, LangChain and Databricks Integration (Databricks Vector Search). Run this before the chapter walkthroughs.

### ✅ Prerequisites

- Unity Catalog permissions to create a **catalog, schema, and volume** (names come from [`conf/data.yml`](../../conf/data.yml)).
- Run this notebook **first**, before `02_load_vector_index` and before the chapter notebooks.

See the [repository README](../../README.md) for full setup.

## Setup

In [0]:
%pip install -r ../../requirements.txt

In [0]:
dbutils.library.restartPython()

In [0]:
import os
from mlflow.models import ModelConfig
from databricks.sdk import WorkspaceClient

In [0]:
data_conf_path = "../../conf/data.yml"
dataset_names = ["faq_dataset", "booking_records_dataset", "qa_dataset"]

In [0]:
default_uc_conf = ModelConfig(development_config=data_conf_path).get("default_uc")

## Create Unity Catalog Schema and Volumes

Create the catalog, schema, and volume that will hold the Unity Airways datasets and tables used by later chapters. _(see Ch 4, "Unstructured Data Preparation for Retrieval")_

In [0]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {default_uc_conf['catalog']}")

In [0]:
spark.sql(f"USE CATALOG {default_uc_conf['catalog']}")

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {default_uc_conf['schema']}")

## Upload Dataset to UC Volumes

Copy the local FAQ, booking, and QA datasets into the Unity Catalog volume so they can be read into tables. _(see Ch 4, "Unstructured Data Preparation for Retrieval")_

In [0]:
spark.sql(f"CREATE VOLUME IF NOT EXISTS {default_uc_conf['schema']}.{default_uc_conf['volume']}")

In [0]:
def upload_files_to_volumes(data_conf_path, dataset_name):
    dataset_conf = ModelConfig(development_config=data_conf_path).get("dataset")
    main_directory = os.path.dirname(os.path.dirname(os.getcwd()))

    local_path = dataset_conf.get(dataset_name).get("local_path")
    full_local_path = os.path.join(main_directory, local_path)

    full_uc_path = dataset_conf.get(dataset_name).get("full_path")

    w = WorkspaceClient()
    with open(full_local_path, "rb") as f:
        binary_data = f.read()

    w.files.upload(full_uc_path, binary_data, overwrite=True)
    
    return full_uc_path

In [0]:
for dataset_name in dataset_names:
    uploaded_path = upload_files_to_volumes(data_conf_path, dataset_name)
    print(f"Uploaded {dataset_name} to: {uploaded_path}")

## Load Dataset as Tables

Read the uploaded files into Delta tables, enabling Change Data Feed on the FAQ table so it can drive a Delta-synced vector index. _(see Ch 4, "Unstructured Data Preparation for Retrieval")_

In [0]:
def create_input_table(data_conf_path, dataset_name, enable_cdf=False):
    dataset_conf = ModelConfig(development_config=data_conf_path).get("dataset")
    tables_conf = ModelConfig(development_config=data_conf_path).get("tables")
    
    # Read csv
    dataset_path = dataset_conf.get(dataset_name).get("full_path")
    dataset_schema = dataset_conf.get(dataset_name).get("dataset_schema")

    df = spark.read.parquet(dataset_path)

    # Save as table
    table_name = tables_conf.get(dataset_name).get("full_path")
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(table_name)
    if enable_cdf:
        spark.sql(f"ALTER TABLE {table_name} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")
    
    return table_name

In [0]:
for dataset_name in dataset_names:
    enable_cdf = True if dataset_name == "faq_dataset" else False
    created_table_name = create_input_table(data_conf_path, dataset_name, enable_cdf=enable_cdf)
    print(f"Created table for {dataset_name}. Table name is {created_table_name}")

## Further Process FAQ Dataset

Combine each question and answer into a single `search_text` column that the embedding model will index for retrieval. _(see Ch 4, "Unstructured Data Preparation for Retrieval")_

In [0]:
from pyspark.sql.functions import concat, lit, col

tables_conf = ModelConfig(development_config=data_conf_path).get("tables")
faq_table = tables_conf.get("faq_dataset").get("full_path")
faq_df = spark.table(faq_table)

faq_df = faq_df.withColumn(
    "search_text",
    concat(lit("question: "), col("question"), lit("\n\nanswer: "), col("answer")),
)

display(faq_df)

In [0]:
faq_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(faq_table)